[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AImotion-Bavaria/kipro_wi/blob/main/lecture2/aggregation_ortools.ipynb)

# Aggregation over Arrays with OR-Tools

`sum`, `max` and `min` over an array of decision variables.

| Index (MiniZinc) | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Index (Python) | 0 | 1 | 2 | 3 | 4 |
| Value | 3 | 2 | 4 | 5 | 3 |
| Domain | 2..6 | 2..6 | 2..6 | 2..6 | 2..6 |

In [ ]:
!pip install -q ortools

## Model

```minizinc
array[1..5] of var 2..6: values;
% fix the values for demonstration
constraint values = [3,2,4,5,3];
```

In Python the array is a list, so the indices run from **0 to 4**.

In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

values = [model.new_int_var(2, 6, f"values[{i}]") for i in range(5)]

# fix the values for demonstration
for i, v in enumerate([3, 2, 4, 5, 3]):
    model.add(values[i] == v)

## Aggregations

* `sum` works directly with Python's `sum()`: it builds a linear expression.
* Python's `max()` and `min()` do **not** work on solver variables
  (they would have to compare values that are only known after solving).
  Instead we create a new variable and link it with `add_max_equality` / `add_min_equality`.
* A `where` clause becomes an `if` in the Python generator expression.

In [ ]:
# MiniZinc: sum(values)
total = sum(values)

# MiniZinc: max(values)
largest = model.new_int_var(2, 6, "largest")
model.add_max_equality(largest, values)

# MiniZinc: min(values)
smallest = model.new_int_var(2, 6, "smallest")
model.add_min_equality(smallest, values)

# MiniZinc: sum(i in 1..5 where i mod 2 == 0) (values[i])
# MiniZinc positions 2 and 4 are Python indices 1 and 3 -> i % 2 == 1 !
even_positions = sum(values[i] for i in range(5) if i % 2 == 1)

## Solve and output

```minizinc
output ["\(sum(values)) \n " ++
        "\(max(values)) \n " ++
        "\(min(values)) \n " ++
        "\(sum(i in 1..5 where i mod 2 == 0) (values[i]))"];
```

In [ ]:
solver = cp_model.CpSolver()
status = solver.solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(solver.value(total))
    print(solver.value(largest))
    print(solver.value(smallest))
    print(solver.value(even_positions))
else:
    print("=====UNSATISFIABLE=====")

## Watch out: indices start at 0

Copying the condition `i mod 2 == 0` literally selects Python indices 0, 2, 4,
i.e. the 1st, 3rd and 5th element, and gives a different result:

In [ ]:
literal_copy = sum(values[i] for i in range(5) if i % 2 == 0)
print("i % 2 == 0:", solver.value(literal_copy), "(elements 1, 3, 5)")
print("i % 2 == 1:", solver.value(even_positions), "(elements 2, 4, as in MiniZinc)")